# 02 - Modelos clásicos de Machine Learning

Se comparan **Regresión Logística, Árbol de Decisión, Random Forest y Gradient Boosting**. Se excluyen variables que generan fuga de información: `puntaje_riesgo`, `nivel_riesgo` y `fraude`.


In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

BASE_DIR = Path.cwd().parent
df = pd.read_csv(BASE_DIR / "datos" / "dataset_fraude_yape.csv")
df["fecha_hora"] = pd.to_datetime(df["fecha_hora"])
df["hora"] = df["fecha_hora"].dt.hour
df["dia_semana"] = df["fecha_hora"].dt.dayofweek

X = df.drop(columns=["id_transaccion", "fecha_hora", "puntaje_riesgo", "nivel_riesgo", "fraude"])
y = df["fraude"]
X = pd.get_dummies(X, columns=["producto"], dtype=float).fillna(0).astype(float)

print("Variables:", X.shape[1])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train:", X_train.shape, "Test:", X_test.shape)


In [ ]:
modelos = {
    "Regresión Logística": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "Árbol de Decisión": DecisionTreeClassifier(max_depth=8, min_samples_split=10, min_samples_leaf=5,
                                                class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=250, max_depth=12, min_samples_split=5,
                                            min_samples_leaf=2, class_weight="balanced",
                                            random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                     max_depth=5, min_samples_split=10, random_state=42)
}

resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train_s, y_train)
    pred = modelo.predict(X_test_s)
    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0)
    })

comparacion = pd.DataFrame(resultados).sort_values("F1", ascending=False)
display(comparacion.round(4))


In [ ]:
mejor_nombre = comparacion.iloc[0]["Modelo"]
mejor_modelo = modelos[mejor_nombre]
pred = mejor_modelo.predict(X_test_s)

print("Mejor modelo:", mejor_nombre)
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, pred))
print("\nReporte:")
print(classification_report(y_test, pred, target_names=["Normal", "Fraude"]))


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test, pred, display_labels=["Normal", "Fraude"])
plt.title(f"Matriz de confusión - {mejor_nombre}")
plt.show()


In [ ]:
MODELOS_DIR = BASE_DIR / "modelos"
MODELOS_DIR.mkdir(exist_ok=True)
joblib.dump(mejor_modelo, MODELOS_DIR / "modelo_fraude.pkl")
joblib.dump(scaler, MODELOS_DIR / "scaler_fraude.pkl")
joblib.dump(X.columns.tolist(), MODELOS_DIR / "columnas_modelo.pkl")
print("Modelo, scaler y columnas guardados.")


## Resultado de referencia

En la ejecución del proyecto, **Gradient Boosting** obtuvo aproximadamente **98.65% Accuracy, 98.29% Precision, 80.43% Recall y 88.47% F1**, siendo el mejor modelo por F1.
